# AgroPest-12 Part B: Standalone BYOL Pretraining for YOLOv12m

**Model:** random `yolo12m.yaml` backbone  
**Input:** `ssl_pool/images` only, 9,180 images  
**Runtime:** Kaggle T4 x2 through PyTorch Distributed Data Parallel (DDP)

This notebook implements BYOL directly with PyTorch and TorchVision. It has no `ssl-detection-lab`, `ssldet`, Lightly, or PyTorch Lightning dependency. Only the online YOLO backbone is transferred to the paired detector notebook.

## Method And Protocol

BYOL forms two independently augmented views of every SSL-pool image. The online encoder, projector, and predictor learn by gradient descent to predict the stop-gradient target encoder and projector representation from the other view. The target branch is initialized as a copy of the online branch and then updated only by exponential moving average (EMA). This predictor and EMA asymmetry avoids the need for negative pairs.

The notebook constructs only `ssl_pool/images`; it neither constructs nor reads label, validation, or test paths. Diagnostics use a fixed image-only SSL-pool sample, so they are exploratory rather than detection metrics.

The fixed seed-445, augmentation-family partition is documented by the attached `partition_manifest.csv`, `class_distribution_report.csv`, and `partition_report.json`. They provide reproducibility provenance only; this label-free notebook does not inspect them.

In [ ]:
# Kaggle already provides CUDA-matched PyTorch. Do not install another torch wheel.
%pip install -q --upgrade "ultralytics>=8.4.96" "scikit-learn>=1.5" seaborn

## Configuration

Every value is explicit and commented for the viva. `per_gpu_batch` counts source images. Each source generates two views and BYOL makes two online and two no-gradient target backbone passes.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import json


@dataclass
class CFG:
    # Dataset and reproducibility
    partb_root: str = "/kaggle/input/datasets/sadikshahriar/agropest-12-partb-safe/AgroPest-12-PartB-Safe"  # Published leakage-safe Part B dataset.
    output_dir: str = "/kaggle/working/partB_02a_byol_yolo12m"  # Writable Kaggle output; publish this directory.
    seed: int = 445  # Shared Part B seed for deterministic DDP sampling and diagnostics.

    # Strict label-free encoder and runtime
    model_yaml: str = "yolo12m.yaml"  # YAML gives random weights; yolo12m.pt is never used for BYOL.
    image_size: int = 224  # Efficient 640-to-224 global SSL view; preserves prevalent large pest morphology on T4s.
    epochs: int = 30  # Fixed Part B SSL budget.
    per_gpu_batch: int = 128  # Validated 224px source-image batch per T4; BYOL does not require contrastive negatives.
    workers: int = 2  # Conservative workers per DDP rank for Kaggle CPU resources.
    sync_batchnorm: bool = True  # Synchronizes online BatchNorm statistics across both GPUs.
    amp: bool = True  # FP16 autocast improves T4 throughput and fits the online plus EMA target networks.
    amp_init_scale: float = 4096.0  # Lower-than-default loss scale avoids first-step FP16 overflow from random BYOL heads.

    # Optimizer and schedules
    learning_rate: float = 3e-4  # Conservative AdamW peak LR for random YOLOv12m SSL.
    min_learning_rate: float = 3e-6  # Cosine-decay floor, 100x below peak LR.
    warmup_epochs: int = 1  # Avoids an unstable first update from random initialization.
    warmup_start_factor: float = 0.10  # Warm-up starts at 10% rather than zero.
    weight_decay: float = 1e-4  # Regularizes online encoder and temporary BYOL heads.
    gradient_clip_norm: float = 5.0  # Bounds rare AMP gradient spikes before every optimizer step.
    base_momentum: float = 0.99  # Faster target tracking than 0.996 for this short, from-scratch 30-epoch run.

    # BYOL heads: never transferred to the detector
    projector_hidden_dim: int = 512  # Compact hidden width for 512-D pooled YOLO features.
    predictor_hidden_dim: int = 512  # Same width for the online-only asymmetric predictor.
    projection_dim: int = 128  # Small pretext embedding that is discarded after SSL.

    # Conservative agricultural two-view augmentations
    crop_scale_min: float = 0.50  # Keeps at least half of a 640px scene so crops rarely remove the pest entirely.
    horizontal_flip_probability: float = 0.50  # Pest class identity is not left/right dependent.
    color_jitter_probability: float = 0.80  # Learns robustness to common outdoor illumination changes.
    color_jitter_brightness: float = 0.35  # Moderate brightness variation.
    color_jitter_contrast: float = 0.35  # Moderate contrast variation.
    color_jitter_saturation: float = 0.20  # Preserves useful insect colour cues better than aggressive jitter.
    color_jitter_hue: float = 0.06  # Small hue shift avoids implausible pest colours.
    grayscale_probability: float = 0.10  # Limited colour removal retains colour information in most views.
    blur_probability: float = 0.50  # Robustness to focus and camera blur in crop scenes.

    # Image-only representation diagnostics
    diagnostic_images: int = 500  # Fixed SSL-pool sample for readable t-SNE and efficient inference.
    retrieval_queries: int = 4  # Four qualitative query rows.
    retrieval_neighbors: int = 5  # Assignment-required top-five cosine neighbours.
    diagnostic_batch_size: int = 64  # Efficient fixed-sample encoder batch on one T4.
    diagnostic_workers: int = 2  # Keeps diagnostic loading within Kaggle CPU limits.
    tsne_perplexity: float = 30.0  # Standard neighborhood scale, capped for the available sample size.
    tsne_iterations: int = 1000  # Enough optimization steps for a stable exploratory projection.
    figure_dpi: int = 180  # Readable exported figures without excessive Kaggle output size.
    resume: bool = False  # Set True only to resume last_ssl.pt with identical configuration.


cfg = CFG()
PARTB_ROOT = Path(cfg.partb_root)
SSL_IMAGES = PARTB_ROOT / "ssl_pool" / "images"
OUTPUT_DIR = Path(cfg.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = asdict(cfg)
(OUTPUT_DIR / "byol_config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(json.dumps(CONFIG, indent=2))

## BYOL Architecture

The random YAML-defined YOLOv12m backbone is the online encoder. Its pooled features pass through a projector and online-only predictor. A frozen copy of the encoder and projector forms the target branch; it receives no gradients and is updated only by EMA after each successful optimizer step.

For each source image, two independently augmented 224px views are created. The loss symmetrically predicts target view B from online view A and target view A from online view B. The projector and predictor are pretext-only modules and are excluded from detector transfer.

## Standalone BYOL Core

The helper module preserves the actual YAML-defined YOLO backbone graph, including each layer's `f`, `i`, and `save` routing. It implements the BYOL heads, symmetric normalized regression loss, EMA target updates, DDP loop, resumable checkpoints, and strict 100% backbone export.

In [ ]:
from textwrap import dedent

CORE_MODULE = OUTPUT_DIR / "byol_core.py"
CORE_MODULE.write_text(dedent('\nimport copy\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset\nfrom torch.utils.data.distributed import DistributedSampler\nfrom torchvision import transforms\nfrom ultralytics import YOLO\n\nIMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef list_ssl_images(image_dir):\n    paths = sorted(\n        path for path in Path(image_dir).iterdir()\n        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES\n    )\n    if len(paths) != 9180:\n        raise ValueError(f"Expected 9,180 SSL-pool images, found {len(paths)}")\n    return paths\n\n\ndef build_byol_transform(config):\n    # Keep crops conservative so views retain both the pest and agricultural context.\n    return transforms.Compose([\n        transforms.RandomResizedCrop(\n            config["image_size"], scale=(config["crop_scale_min"], 1.0),\n            ratio=(0.75, 1.333333),\n        ),\n        transforms.RandomHorizontalFlip(p=config["horizontal_flip_probability"]),\n        transforms.RandomApply([\n            transforms.ColorJitter(\n                config["color_jitter_brightness"], config["color_jitter_contrast"],\n                config["color_jitter_saturation"], config["color_jitter_hue"],\n            )\n        ], p=config["color_jitter_probability"]),\n        transforms.RandomGrayscale(p=config["grayscale_probability"]),\n        transforms.RandomApply([\n            transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0))\n        ], p=config["blur_probability"]),\n        transforms.ToTensor(),\n        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n    ])\n\n\nclass ImagePairDataset(Dataset):\n    def __init__(self, paths, transform):\n        self.paths = list(paths)\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, index):\n        with Image.open(self.paths[index]) as image:\n            image = image.convert("RGB")\n            return self.transform(image), self.transform(image)\n\n\nclass YOLOBackboneEncoder(nn.Module):\n    """Run the YAML-defined Ultralytics backbone with its original graph routing."""\n    def __init__(self, yolo_task_model):\n        super().__init__()\n        backbone = getattr(yolo_task_model, "yaml", {}).get("backbone", [])\n        if not backbone:\n            raise ValueError("The Ultralytics model has no YAML backbone definition.")\n        self.backbone_end = len(backbone)\n        self.layers = nn.ModuleList(list(yolo_task_model.model[:self.backbone_end]))\n        self.save = set(getattr(yolo_task_model, "save", []))\n\n    def forward_feature_map(self, x):\n        saved = []\n        for module in self.layers:\n            source = getattr(module, "f", -1)\n            if source != -1:\n                x = saved[source] if isinstance(source, int) else [\n                    x if item == -1 else saved[item] for item in source\n                ]\n            x = module(x)\n            saved.append(x if getattr(module, "i", -1) in self.save else None)\n        if not isinstance(x, torch.Tensor):\n            raise TypeError("Final YOLO backbone output must be a tensor.")\n        return x\n\n    def forward(self, x):\n        return F.adaptive_avg_pool2d(self.forward_feature_map(x), 1).flatten(1)\n\n\ndef mlp(input_dim, hidden_dim, output_dim, final_bn):\n    layers = [\n        nn.Linear(input_dim, hidden_dim, bias=False),\n        nn.BatchNorm1d(hidden_dim),\n        nn.ReLU(inplace=True),\n        nn.Linear(hidden_dim, output_dim, bias=not final_bn),\n    ]\n    if final_bn:\n        layers.append(nn.BatchNorm1d(output_dim, affine=False))\n    return nn.Sequential(*layers)\n\n\nclass BYOLModel(nn.Module):\n    def __init__(self, encoder, feature_dim, config):\n        super().__init__()\n        self.encoder = encoder\n        self.projector = mlp(\n            feature_dim, config["projector_hidden_dim"], config["projection_dim"],\n            final_bn=True,\n        )\n        self.predictor = mlp(\n            config["projection_dim"], config["predictor_hidden_dim"],\n            config["projection_dim"], final_bn=False,\n        )\n        self.target_encoder = copy.deepcopy(self.encoder)\n        self.target_projector = copy.deepcopy(self.projector)\n        for module in (self.target_encoder, self.target_projector):\n            module.requires_grad_(False)\n            module.eval()\n\n    def online(self, images):\n        return self.predictor(self.projector(self.encoder(images)))\n\n    def forward(self, view_a, view_b):\n        # DDP must invoke this method rather than model.module so gradient hooks synchronize.\n        online_predictions = self.online(torch.cat([view_a, view_b], dim=0))\n        prediction_a, prediction_b = online_predictions.chunk(2, dim=0)\n        target_projections = self.target(torch.cat([view_a, view_b], dim=0))\n        target_a, target_b = target_projections.chunk(2, dim=0)\n        return 0.5 * (\n            negative_cosine_loss(prediction_a, target_b)\n            + negative_cosine_loss(prediction_b, target_a)\n        )\n\n    @torch.no_grad()\n    def target(self, images):\n        return self.target_projector(self.target_encoder(images))\n\n    @torch.no_grad()\n    def update_target(self, momentum):\n        # EMA parameters; copying buffers keeps target BN statistics synchronized and stable.\n        for online, target in ((self.encoder, self.target_encoder), (self.projector, self.target_projector)):\n            for online_parameter, target_parameter in zip(online.parameters(), target.parameters()):\n                target_parameter.data.mul_(momentum).add_(online_parameter.data, alpha=1.0 - momentum)\n            for online_buffer, target_buffer in zip(online.buffers(), target.buffers()):\n                target_buffer.copy_(online_buffer)\n\n\ndef negative_cosine_loss(prediction, target):\n    prediction = F.normalize(prediction, dim=1)\n    target = F.normalize(target.detach(), dim=1)\n    return 2.0 - 2.0 * (prediction * target).sum(dim=1).mean()\n\n\ndef byol_loss(model, view_a, view_b):\n    online_predictions = model.online(torch.cat([view_a, view_b], dim=0))\n    prediction_a, prediction_b = online_predictions.chunk(2, dim=0)\n    target_projections = model.target(torch.cat([view_a, view_b], dim=0))\n    target_a, target_b = target_projections.chunk(2, dim=0)\n    return 0.5 * (\n        negative_cosine_loss(prediction_a, target_b)\n        + negative_cosine_loss(prediction_b, target_a)\n    )\n\n\ndef learning_rate_for_step(config, step, total_steps, warmup_steps):\n    if step < warmup_steps:\n        progress = (step + 1) / max(1, warmup_steps)\n        factor = config["warmup_start_factor"] + (1.0 - config["warmup_start_factor"]) * progress\n        return config["learning_rate"] * factor\n    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps - 1)\n    cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))\n    return config["min_learning_rate"] + (config["learning_rate"] - config["min_learning_rate"]) * cosine\n\n\ndef target_momentum_for_step(config, step, total_steps):\n    progress = min(step / max(1, total_steps - 1), 1.0)\n    return 1.0 - (1.0 - config["base_momentum"]) * (math.cos(math.pi * progress) + 1.0) / 2.0\n\n\ndef cpu_state_dict(module):\n    return {key: value.detach().cpu() for key, value in module.state_dict().items()}\n\n\ndef save_epoch_state(output_dir, epoch, best_loss, model, optimizer, scaler, feature_dim, config):\n    state = {\n        "format_version": 1,\n        "epoch": epoch,\n        "best_loss": best_loss,\n        "yolo_model": config["model_yaml"],\n        "feature_dim": feature_dim,\n        "encoder_state_dict": cpu_state_dict(model.encoder),\n        "projector_state_dict": cpu_state_dict(model.projector),\n        "predictor_state_dict": cpu_state_dict(model.predictor),\n        "target_encoder_state_dict": cpu_state_dict(model.target_encoder),\n        "target_projector_state_dict": cpu_state_dict(model.target_projector),\n        "optimizer_state_dict": optimizer.state_dict(),\n        "scaler_state_dict": scaler.state_dict(),\n        "config": config,\n    }\n    torch.save(state, output_dir / "last_ssl.pt")\n    return state\n\n\ndef load_online_state(model, state):\n    model.encoder.load_state_dict(state["encoder_state_dict"], strict=True)\n    model.projector.load_state_dict(state["projector_state_dict"], strict=True)\n    model.predictor.load_state_dict(state["predictor_state_dict"], strict=True)\n    model.target_encoder.load_state_dict(state["target_encoder_state_dict"], strict=True)\n    model.target_projector.load_state_dict(state["target_projector_state_dict"], strict=True)\n\n\ndef export_best_backbone(output_dir, config):\n    best_state = torch.load(output_dir / "best_ssl.pt", map_location="cpu", weights_only=False)\n    detector = YOLO(config["model_yaml"])\n    encoder = YOLOBackboneEncoder(detector.model)\n    encoder.load_state_dict(best_state["encoder_state_dict"], strict=True)\n    backbone_state = cpu_state_dict(encoder)\n    torch.save({\n        "format_version": 1,\n        "method": "BYOL",\n        "yolo_model": config["model_yaml"],\n        "encoder_state_dict": backbone_state,\n        "source_ssl_checkpoint": str(output_dir / "best_ssl.pt"),\n    }, output_dir / "byol_best_backbone_yolo12m.pt")\n\n    target_state = detector.model.state_dict()\n    mapped_state = {}\n    for key, value in backbone_state.items():\n        layer_key, suffix = key.split(".", 2)[1:]\n        detector_key = f"model.{layer_key}.{suffix}"\n        if detector_key not in target_state or target_state[detector_key].shape != value.shape:\n            raise RuntimeError(f"Cannot map backbone key {key} to detector key {detector_key}")\n        mapped_state[detector_key] = value\n    coverage = len(mapped_state) / len(backbone_state)\n    if coverage != 1.0:\n        raise RuntimeError(f"Backbone export coverage must be 100%, got {coverage:.1%}")\n    detector.model.load_state_dict(mapped_state, strict=False)\n    detector.model.args = {"model": config["model_yaml"], "task": "detect"}\n    torch.save({\n        "model": copy.deepcopy(detector.model).half(),\n        "ema": None,\n        "optimizer": None,\n        "epoch": best_state["epoch"],\n        "train_args": detector.model.args,\n    }, output_dir / "byol_best_backbone_yolo12m_detector.pt")\n    report = {\n        "method": "BYOL",\n        "source": str(output_dir / "best_ssl.pt"),\n        "target_model": config["model_yaml"],\n        "backbone_layers": encoder.backbone_end,\n        "matched_keys": len(mapped_state),\n        "target_keys": len(backbone_state),\n        "coverage": coverage,\n        "projector_transferred": False,\n        "predictor_transferred": False,\n        "detector_neck_head_transferred": False,\n    }\n    (output_dir / "transfer_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    return report\n\n\ndef build_byol_model(config, device):\n    detector = YOLO(config["model_yaml"]).model.to(device)\n    encoder = YOLOBackboneEncoder(detector).to(device)\n    with torch.no_grad():\n        feature_dim = encoder(torch.zeros(2, 3, config["image_size"], config["image_size"], device=device)).shape[1]\n    return BYOLModel(encoder, feature_dim, config).to(device), feature_dim\n\n\ndef train_distributed(config):\n    local_rank = int(os.environ["LOCAL_RANK"])\n    rank = int(os.environ["RANK"])\n    world_size = int(os.environ["WORLD_SIZE"])\n    torch.cuda.set_device(local_rank)\n    device = torch.device("cuda", local_rank)\n    dist.init_process_group(backend="nccl")\n    seed_everything(config["seed"] + rank)\n    output_dir = Path(config["output_dir"])\n    paths = list_ssl_images(Path(config["partb_root"]) / "ssl_pool" / "images")\n    dataset = ImagePairDataset(paths, build_byol_transform(config))\n    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=config["seed"], drop_last=True)\n    loader = DataLoader(dataset, batch_size=config["per_gpu_batch"], sampler=sampler, num_workers=config["workers"], pin_memory=True, persistent_workers=config["workers"] > 0, drop_last=True)\n    model, feature_dim = build_byol_model(config, device)\n    if config["sync_batchnorm"]:\n        model = nn.SyncBatchNorm.convert_sync_batchnorm(model)\n    model = DDP(model, device_ids=[local_rank], output_device=local_rank, broadcast_buffers=False)\n    optimizer = torch.optim.AdamW(\n        [parameter for parameter in model.parameters() if parameter.requires_grad],\n        lr=config["learning_rate"], weight_decay=config["weight_decay"],\n    )\n    scaler = torch.amp.GradScaler("cuda", enabled=config["amp"], init_scale=config["amp_init_scale"])\n    best_loss, start_epoch, history = float("inf"), 0, []\n    history_path = output_dir / "history.csv"\n    last_path = output_dir / "last_ssl.pt"\n    if config["resume"] and last_path.is_file():\n        state = torch.load(last_path, map_location="cpu", weights_only=False)\n        load_online_state(model.module, state)\n        optimizer.load_state_dict(state["optimizer_state_dict"])\n        scaler.load_state_dict(state["scaler_state_dict"])\n        best_loss, start_epoch = state["best_loss"], state["epoch"] + 1\n        if rank == 0 and history_path.is_file():\n            with history_path.open(newline="", encoding="utf-8") as handle:\n                history = list(csv.DictReader(handle))\n    steps_per_epoch = len(loader)\n    total_steps = steps_per_epoch * config["epochs"]\n    warmup_steps = config["warmup_epochs"] * steps_per_epoch\n    for epoch in range(start_epoch, config["epochs"]):\n        sampler.set_epoch(epoch)\n        model.train()\n        model.module.target_encoder.eval()\n        model.module.target_projector.eval()\n        started = time.perf_counter()\n        loss_sum = torch.zeros(1, device=device)\n        examples = torch.zeros(1, device=device)\n        epoch_start_lr = learning_rate_for_step(config, epoch * steps_per_epoch, total_steps, warmup_steps)\n        epoch_start_momentum = target_momentum_for_step(config, epoch * steps_per_epoch, total_steps)\n        for batch_index, (view_a, view_b) in enumerate(loader):\n            step = epoch * steps_per_epoch + batch_index\n            lr = learning_rate_for_step(config, step, total_steps, warmup_steps)\n            momentum = target_momentum_for_step(config, step, total_steps)\n            for group in optimizer.param_groups:\n                group["lr"] = lr\n            view_a = view_a.to(device, non_blocking=True)\n            view_b = view_b.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=config["amp"]):\n                loss = model(view_a, view_b)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Non-finite loss at epoch {epoch + 1}, batch {batch_index + 1}: {loss.item()}")\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            gradients_finite = all(\n                parameter.grad is None or torch.isfinite(parameter.grad).all()\n                for parameter in model.parameters()\n            )\n            if gradients_finite:\n                gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip_norm"])\n                if not torch.isfinite(gradient_norm):\n                    raise FloatingPointError(f"Non-finite clipped gradient norm at epoch {epoch + 1}, batch {batch_index + 1}")\n                scaler.step(optimizer)\n                model.module.update_target(momentum)\n            elif rank == 0:\n                print(f"AMP overflow at epoch {epoch + 1}, batch {batch_index + 1}; skipped update and reduced loss scale.")\n            # update() observes inf values registered by unscale_ and lowers the next loss scale.\n            scaler.update()\n            loss_sum += loss.detach() * view_a.shape[0]\n            examples += view_a.shape[0]\n        dist.all_reduce(loss_sum, op=dist.ReduceOp.SUM)\n        dist.all_reduce(examples, op=dist.ReduceOp.SUM)\n        epoch_loss = (loss_sum / examples).item()\n        if rank == 0:\n            history.append({\n                "epoch": epoch + 1,\n                "loss": epoch_loss,\n                "learning_rate_start": epoch_start_lr,\n                "learning_rate_end": lr,\n                "ema_momentum_start": epoch_start_momentum,\n                "ema_momentum_end": momentum,\n                "seconds": time.perf_counter() - started,\n            })\n            with history_path.open("w", newline="", encoding="utf-8") as handle:\n                writer = csv.DictWriter(handle, fieldnames=history[0].keys())\n                writer.writeheader()\n                writer.writerows(history)\n            state = save_epoch_state(output_dir, epoch, min(best_loss, epoch_loss), model.module, optimizer, scaler, feature_dim, config)\n            if epoch_loss < best_loss:\n                best_loss = epoch_loss\n                torch.save(state, output_dir / "best_ssl.pt")\n            print(f"Epoch {epoch + 1:02d}/{config[\'epochs\']} | loss={epoch_loss:.5f} | lr={lr:.2e} | ema={momentum:.6f}")\n        dist.barrier()\n    if rank == 0:\n        report = export_best_backbone(output_dir, config)\n        manifest = {\n            "method": "BYOL",\n            "strict_label_free": True,\n            "ssl_images": len(paths),\n            "world_size": world_size,\n            "local_source_batch": config["per_gpu_batch"],\n            "views_per_source": 2,\n            "online_backbone_passes_per_source": 2,\n            "target_backbone_passes_per_source": 2,\n            "best_loss": best_loss,\n            "outputs": {\n                "best_ssl": "best_ssl.pt",\n                "last_ssl": "last_ssl.pt",\n                "history": "history.csv",\n                "backbone": "byol_best_backbone_yolo12m.pt",\n                "detector": "byol_best_backbone_yolo12m_detector.pt",\n                "transfer_report": "transfer_report.json",\n            },\n            "config": config,\n        }\n        (output_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n        print(json.dumps(report, indent=2))\n    dist.barrier()\n    dist.destroy_process_group()\n'), encoding="utf-8")
RUNNER = OUTPUT_DIR / "train_byol.py"
RUNNER.write_text(
    "from pathlib import Path\nimport json\nfrom byol_core import train_distributed\n"
    "config = json.loads(Path(__file__).with_name('byol_config.json').read_text(encoding='utf-8'))\n"
    "train_distributed(config)\n",
    encoding="utf-8",
)
print(f"Wrote standalone core: {CORE_MODULE}")
print(f"Wrote DDP launcher: {RUNNER}")

In [ ]:
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision
import ultralytics
from PIL import Image
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

sys.path.insert(0, str(OUTPUT_DIR))
from byol_core import (
    IMAGENET_MEAN, IMAGENET_STD, ImagePairDataset, YOLOBackboneEncoder,
    build_byol_model, build_byol_transform, byol_loss, list_ssl_images,
)

assert sys.version_info >= (3, 10), "Python 3.10+ is required."
assert tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2, 4), "PyTorch 2.4+ is required."
assert tuple(map(int, torchvision.__version__.split("+")[0].split(".")[:2])) >= (0, 19), "Torchvision 0.19+ is required."
assert tuple(map(int, ultralytics.__version__.split(".")[:3])) >= (8, 4, 96), "Ultralytics 8.4.96+ is required."
assert SSL_IMAGES.is_dir(), f"Attach AgroPest-12-PartB-Safe. Missing: {SSL_IMAGES}"
assert not any(SSL_IMAGES.glob("*.txt")), "The SSL input must not contain label files."
ssl_paths = list_ssl_images(SSL_IMAGES)
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "Select Kaggle GPU T4 x2 before training."
gpu_names = [torch.cuda.get_device_name(index) for index in range(2)]
assert all("T4" in name for name in gpu_names), f"Expected two Kaggle T4 GPUs, found: {gpu_names}"
gpu_memory_gib = [round(torch.cuda.get_device_properties(index).total_memory / 2**30, 1) for index in range(2)]
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed); torch.cuda.manual_seed_all(cfg.seed)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid")

pd.Series({
    "Python": sys.version.split()[0], "PyTorch": torch.__version__, "Torchvision": torchvision.__version__, "Ultralytics": ultralytics.__version__,
    "GPU count": torch.cuda.device_count(), "GPU 0": gpu_names[0], "GPU 0 memory (GiB)": gpu_memory_gib[0],
    "GPU 1": gpu_names[1], "GPU 1 memory (GiB)": gpu_memory_gib[1], "SSL images": len(ssl_paths), "label files read": 0,
    "source images per DDP step": cfg.per_gpu_batch * 2, "views per local BYOL step": 2 * cfg.per_gpu_batch,
    "backbone passes per local source": 4,
})

## Augmentation Preview And Finite-Loss Sanity Check

Inspect the two views before training: each should differ in crop, lighting, or blur while retaining enough common pest and scene content. The preflight builds the exact random BYOL model, checks finite symmetric loss and online gradients, and confirms the EMA target remains gradient-free.

In [ ]:
preview_dataset = ImagePairDataset(ssl_paths, build_byol_transform(CONFIG))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(4, 2, figsize=(8, 14))
for row, index in enumerate(random.Random(323).sample(range(len(preview_dataset)), 4)):
    for column, view in enumerate(preview_dataset[index]):
        axes[row, column].imshow((view * std + mean).clamp(0, 1).permute(1, 2, 0))
        axes[row, column].axis("off")
    axes[row, 0].set_ylabel(ssl_paths[index].name[:30], fontsize=8)
axes[0, 0].set_title("Online view"); axes[0, 1].set_title("EMA target view")
plt.tight_layout(); plt.show()

device = torch.device("cuda:0")
sanity_model, feature_dim = build_byol_model(CONFIG, device)
sanity_a, sanity_b = next(iter(DataLoader(preview_dataset, batch_size=2, shuffle=False, drop_last=True)))
with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=cfg.amp):
    sanity_loss = byol_loss(sanity_model, sanity_a.to(device), sanity_b.to(device))
assert torch.isfinite(sanity_loss), f"Non-finite preflight loss: {sanity_loss.item()}"
sanity_loss.backward()
online_gradient = any(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in sanity_model.encoder.parameters())
target_gradient = any(parameter.grad is not None for parameter in sanity_model.target_encoder.parameters())
assert online_gradient, "Online encoder did not receive a finite gradient."
assert not target_gradient, "EMA target encoder must not receive gradients."
print({"backbone_feature_dimension": feature_dim, "finite_loss": sanity_loss.detach().item(), "finite_online_gradient": online_gradient, "target_has_gradient": target_gradient})
del sanity_model
torch.cuda.empty_cache()

## Dual-GPU BYOL Training

The launcher uses `torch.distributed.run` with NCCL, `DistributedSampler`, `DistributedDataParallel`, synchronized BatchNorm, AMP, finite-loss checks, gradient clipping, rank-zero-only writes, and epoch checkpoints. EMA momentum follows a cosine schedule from 0.99 toward 1.0. BYOL has no negatives, so the batch size is selected for stable gradient estimates and T4 throughput rather than negative-pair count.

In [ ]:
import subprocess

subprocess.run([
    sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2",
    str(RUNNER),
], check=True)

## Training Loss And Checkpoint Handoff

The BYOL loss is an SSL optimization signal, not detection accuracy. `best_ssl.pt` is chosen solely by lowest mean training loss; `last_ssl.pt` is resumable. The export report enforces full key and shape coverage for the online backbone and confirms that no projector, predictor, neck, or detection head is transferred.

In [ ]:
from matplotlib.ticker import MaxNLocator

history = pd.read_csv(OUTPUT_DIR / "history.csv")
manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
transfer_report = json.loads((OUTPUT_DIR / "transfer_report.json").read_text(encoding="utf-8"))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history["epoch"], history["loss"], marker="o", linewidth=2.2, color="#1261A0")
axes[0].set(title="BYOL Symmetric Regression Loss", xlabel="Epoch", ylabel="2 - 2 cosine similarity")
axes[1].plot(history["epoch"], history["learning_rate_start"], marker="o", linewidth=2.0, label="Epoch start")
axes[1].plot(history["epoch"], history["learning_rate_end"], marker="o", linewidth=2.0, label="Epoch end")
axes[1].set(title="Linear Warm-up + Cosine Decay", xlabel="Epoch", ylabel="Learning rate"); axes[1].legend()
axes[2].plot(history["epoch"], history["ema_momentum_start"], marker="o", linewidth=2.0, label="Epoch start")
axes[2].plot(history["epoch"], history["ema_momentum_end"], marker="o", linewidth=2.0, label="Epoch end")
axes[2].set(title="Cosine EMA Target Momentum", xlabel="Epoch", ylabel="Momentum"); axes[2].legend()
for axis in axes:
    axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "byol_training_curves.png", dpi=cfg.figure_dpi); plt.show()
pd.Series({
    "best SSL loss": manifest["best_loss"], "best SSL checkpoint": OUTPUT_DIR / "best_ssl.pt",
    "resumable checkpoint": OUTPUT_DIR / "last_ssl.pt", "backbone checkpoint": OUTPUT_DIR / "byol_best_backbone_yolo12m.pt",
    "detector-compatible checkpoint": OUTPUT_DIR / "byol_best_backbone_yolo12m_detector.pt",
    "backbone coverage": f"{transfer_report['coverage']:.1%}",
})

## Image-Only Representation Diagnostics And Retrieval Sanity Check

t-SNE and cosine retrieval use a fixed SSL-pool sample only. A non-zero feature standard deviation and a spread of pairwise cosine similarities are basic anti-collapse checks; they do not prove downstream detection quality. Complete the short interpretation after execution without making label-based claims.

Describe only visible image-level similarity in the nearest-neighbour panels, such as related pest shape, colour, leaf texture, framing, or background. Do not infer class accuracy, mAP, or downstream detector quality from these exploratory diagnostics.

In [ ]:
diagnostic_paths = sorted(random.Random(cfg.seed).sample(ssl_paths, min(cfg.diagnostic_images, len(ssl_paths))))
evaluation_transform = transforms.Compose([
    transforms.Resize(cfg.image_size + 32), transforms.CenterCrop(cfg.image_size),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class SSLFeatureDataset(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return evaluation_transform(image.convert("RGB")), self.paths[index].name

best_state = torch.load(OUTPUT_DIR / "best_ssl.pt", map_location="cpu", weights_only=False)
diagnostic_model, _ = build_byol_model(CONFIG, device)
diagnostic_model.encoder.load_state_dict(best_state["encoder_state_dict"], strict=True)
diagnostic_model.encoder.eval()
feature_loader = DataLoader(SSLFeatureDataset(diagnostic_paths), batch_size=cfg.diagnostic_batch_size, shuffle=False, num_workers=cfg.diagnostic_workers, pin_memory=True)
features, names = [], []
with torch.inference_mode():
    for images, batch_names in feature_loader:
        features.append(torch.nn.functional.normalize(diagnostic_model.encoder(images.to(device, non_blocking=True)), dim=1).cpu())
        names.extend(batch_names)
feature_matrix = torch.cat(features)
assert torch.isfinite(feature_matrix).all()
pairwise = feature_matrix @ feature_matrix.T
off_diagonal = pairwise[~torch.eye(len(feature_matrix), dtype=torch.bool)]
diagnostics = pd.Series({
    "diagnostic images": len(names), "feature dimensions": feature_matrix.shape[1],
    "mean L2 norm": feature_matrix.norm(dim=1).mean().item(),
    "mean feature std": feature_matrix.std(dim=0).mean().item(),
    "mean off-diagonal cosine": off_diagonal.mean().item(), "off-diagonal cosine std": off_diagonal.std().item(),
})
assert diagnostics["mean feature std"] > 0, "Collapsed constant features detected."
display(diagnostics)

In [ ]:
perplexity = min(cfg.tsne_perplexity, max(5.0, (len(feature_matrix) - 1) / 3))
coordinates = TSNE(n_components=2, perplexity=perplexity, learning_rate="auto", init="pca", max_iter=cfg.tsne_iterations, random_state=cfg.seed).fit_transform(feature_matrix.numpy())
fig, axis = plt.subplots(figsize=(9, 7))
axis.scatter(coordinates[:, 0], coordinates[:, 1], s=15, alpha=0.7, color="#7c3aed")
axis.set(title="t-SNE: BYOL YOLOv12m SSL-Pool Features", xlabel="t-SNE 1", ylabel="t-SNE 2")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "byol_tsne_ssl_pool.png", dpi=cfg.figure_dpi); plt.show()
pd.DataFrame({"filename": names, "tsne_1": coordinates[:, 0], "tsne_2": coordinates[:, 1]}).to_csv(OUTPUT_DIR / "ssl_pool_embeddings_tsne.csv", index=False)

query_indices = random.Random(cfg.seed).sample(range(len(diagnostic_paths)), cfg.retrieval_queries)
fig, axes = plt.subplots(cfg.retrieval_queries, cfg.retrieval_neighbors + 1, figsize=(3.2 * (cfg.retrieval_neighbors + 1), 3.2 * cfg.retrieval_queries))
retrieval_rows = []
for row, query_index in enumerate(query_indices):
    similarities = feature_matrix @ feature_matrix[query_index]
    neighbors = [index for index in torch.topk(similarities, k=cfg.retrieval_neighbors + 1).indices.tolist() if index != query_index][:cfg.retrieval_neighbors]
    for column, index in enumerate([query_index] + neighbors):
        with Image.open(diagnostic_paths[index]) as image:
            axes[row, column].imshow(image.convert("RGB"))
        axes[row, column].set_title("Query" if column == 0 else f"cos={similarities[index]:.3f}", fontsize=9)
        axes[row, column].axis("off")
        retrieval_rows.append({"query": diagnostic_paths[query_index].name, "rank": column, "match": diagnostic_paths[index].name, "cosine_similarity": float(similarities[index])})
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "byol_nearest_neighbors.png", dpi=cfg.figure_dpi); plt.show()
pd.DataFrame(retrieval_rows).to_csv(OUTPUT_DIR / "byol_nearest_neighbors.csv", index=False)
print("After execution, describe only visible image-level structure in the retrieved neighbours; reserve all detection-performance claims for Notebook 02b.")

## Publication And Viva Notes

Publish the complete `/kaggle/working/partB_02a_byol_yolo12m/` directory as a Kaggle dataset. Notebook 02b attaches it and uses the backbone checkpoint plus `transfer_report.json`.

- Online branch: encoder, projector, and predictor trained by AdamW.
- Target branch: encoder and projector initialized from online weights, updated only by EMA.
- Objective: average normalized regression from online view A to target view B and online view B to target view A.
- The target branch receives no gradients; the predictor is intentionally online-only.
- Projector, predictor, optimizer, scheduler, detector neck, and detector head are never transferred.
- BYOL uses no contrastive negatives. Its batch is chosen for T4 efficiency and gradient stability, not a negative count.